In [1]:
import sys
!{sys.executable} -m pip install scikit-learn joblib


   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   --------------- ------------------------ 3.1/8.0 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 20.7 MB/s  0:00:00
   ---------------------------------------- 0.0/36.3 MB ? eta -:--:--
   ----- ---------------------------------- 5.2/36.3 MB 26.6 MB/s eta 0:00:02
   ------------ --------------------------- 11.5/36.3 MB 27.8 MB/s eta 0:00:01
   ------------------- -------------------- 17.6/36.3 MB 29.1 MB/s eta 0:00:01
   ------------------------- -------------- 23.6/36.3 MB 28.7 MB/s eta 0:00:01
   -------------------------------- ------- 29.6/36.3 MB 28.9 MB/s eta 0:00:01
   ---------------------------------------  35.9/36.3 MB 29.2 MB/s eta 0:00:01
   ---------------------------------------- 36.3/36.3 MB 27.5 MB/s  0:00:01

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sklearn
print(sklearn.__version__)


1.8.0


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import joblib


In [4]:
df = pd.read_csv("../_data/processed/final_features.csv")
df.head()


,churn,total_revenue_log,avg_order_value_log,total_quantity_log,total_transactions_log,customer_lifetime_days_log
0,1,11.258774,7.732839,11.215678,2.564949,5.993961
1,0,8.636632,3.146997,8.097731,2.197225,5.998937
2,0,7.611051,3.703671,7.906547,1.791759,5.894403
3,0,8.396085,3.269827,7.393263,1.609438,6.347389
4,1,5.815324,3.028712,5.288267,0.693147,0.000000


In [5]:
X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


In [6]:
lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train, y_train)

y_prob_lr = lr.predict_proba(X_test)[:, 1]
print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, y_prob_lr))


Logistic Regression ROC-AUC: 0.7816455938697318


In [7]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))


Random Forest ROC-AUC: 0.8043994252873563

Confusion Matrix:
 [[591 279]
 [120 480]]

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.68      0.75       870
           1       0.63      0.80      0.71       600

    accuracy                           0.73      1470
   macro avg       0.73      0.74      0.73      1470
weighted avg       0.75      0.73      0.73      1470



In [9]:
joblib.dump(rf, "../_models/churn_model.pkl")


['../_models/churn_model.pkl']

In [11]:
import joblib
import pandas as pd

# Load trained churn model
model = joblib.load("../_models/churn_model.pkl")

# Load customer-level data (with IDs)
customer_df = pd.read_csv("../_data/processed/customer_level_features.csv")

# Load ML-ready features (without IDs)
features_df = pd.read_csv("../_data/processed/final_features.csv")


In [13]:
# Drop target column before prediction
X_full = features_df.drop(columns=["churn"])

customer_df["churn_probability"] = model.predict_proba(X_full)[:, 1]



In [14]:
churn_ranking = customer_df.sort_values(
    by="churn_probability",
    ascending=False
)

churn_ranking.head(10)


,Customer_ID,total_transactions,total_quantity,total_revenue,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,recency_days,churn_probability
3781,16165.0,1,2,79.90,79.90,2010-07-22 14:17:00,2010-07-22 14:17:00,0,504,0.966799
1823,14190.0,1,2,79.90,79.90,2010-02-11 14:27:00,2010-02-11 14:27:00,0,665,0.966799
2533,14906.0,1,1,68.44,68.44,2010-09-20 16:30:00,2010-09-20 16:30:00,0,444,0.963600
223,12570.0,1,1,77.52,77.52,2010-01-26 17:28:00,2010-01-26 17:28:00,0,681,0.960820
5658,18068.0,1,6,101.70,101.70,2011-02-23 14:50:00,2011-02-23 14:50:00,0,288,0.944863
3665,16047.0,1,1,107.14,107.14,2010-01-26 16:34:00,2010-01-26 16:34:00,0,681,0.944631
5167,17570.0,1,1,100.00,100.00,2010-11-23 11:56:00,2010-11-23 11:56:00,0,381,0.941288
3906,16291.0,1,1,117.72,117.72,2010-01-26 16:26:00,2010-01-26 16:26:00,0,681,0.940192
834,13185.0,1,12,71.40,71.40,2011-03-17 11:56:00,2011-03-17 11:56:00,0,267,0.938025
5340,17747.0,1,12,71.40,71.40,2011-08-19 08:13:00,2011-08-19 08:13:00,0,112,0.938025


In [15]:
churn_ranking.to_csv(
    "../_reports/churn_risk_ranking.csv",
    index=False
)
